# Multi-Asset CTA Strategy V2 — Transition Strategy

## 01 — Universe and Data Engine

This notebook establishes the initial V2 cross-asset research universe and data layer.

### Included
- Equity indices
- FX
- Commodities
- Bonds and rates
- Bitcoin

### Excluded for now
- Individual stocks
- Other digital assets

All outputs from this notebook are written to:

`/content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.01/`

In [ ]:
# ============================================================
# 0) INSTALL AND IMPORT
# ============================================================

import sys
import subprocess
import importlib.util

for package in ["yfinance", "pyarrow"]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package]
        )

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import yfinance as yf

warnings.filterwarnings("ignore")

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

print("V2.01 environment ready.")

V2.01 environment ready.


In [ ]:
# ============================================================
# 1) GOOGLE DRIVE AND V2.01 DIRECTORIES
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

PROJECT_PARENT = Path(
    "/content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2"
)

OUTPUT_ROOT = PROJECT_PARENT / "v2.01"

DIRS = {
    "root": OUTPUT_ROOT,
    "data_raw": OUTPUT_ROOT / "data" / "raw",
    "data_processed": OUTPUT_ROOT / "data" / "processed",
    "data_cache": OUTPUT_ROOT / "data" / "cache",
    "results": OUTPUT_ROOT / "results",
    "figures": OUTPUT_ROOT / "figures",
    "manifests": OUTPUT_ROOT / "manifests",
    "config": OUTPUT_ROOT / "config",
}

for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

print("Project parent:", PROJECT_PARENT)
print("V2.01 output root:", OUTPUT_ROOT)

for key, path in DIRS.items():
    print(f"{key:>16}: {path}")

Mounted at /content/drive
Project parent: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2
V2.01 output root: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.01
            root: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.01
        data_raw: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.01/data/raw
  data_processed: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.01/data/processed
      data_cache: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.01/data/cache
         results: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.01/results
         figures: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.01/figures
       manifests: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Str

## Research-date policy

The default research sample ends on **31 December 2025**.

Because V2 is being designed during 2026, 2026 observations should not be described as a pristine final holdout. They can later be used deliberately as a shadow validation period.

A genuinely untouched final holdout should be designated prospectively before the finished strategy is frozen.

In [ ]:
# ============================================================
# 2) RESEARCH DATES AND CONFIGURATION
# ============================================================

DATA_START = "1990-01-01"
DIGITAL_ASSETS_START = "2017-01-01"
RESEARCH_END = "2025-12-31"
ALLOW_POST_2025 = False

# yfinance end date is exclusive.
DOWNLOAD_END = None if ALLOW_POST_2025 else "2026-01-01"

CONFIG = {
    "notebook": "01 - V2 Universe and Data Engine",
    "output_root": str(OUTPUT_ROOT),
    "data_start": DATA_START,
    "digital_assets_start": DIGITAL_ASSETS_START,
    "research_end": RESEARCH_END,
    "allow_post_2025": ALLOW_POST_2025,
}

config_path = DIRS["config"] / "v2_01_config.json"

with config_path.open("w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)

print(json.dumps(CONFIG, indent=2))

{
  "notebook": "01 - V2 Universe and Data Engine",
  "output_root": "/content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.01",
  "data_start": "1990-01-01",
  "digital_assets_start": "2017-01-01",
  "research_end": "2025-12-31",
  "allow_post_2025": false
}


## Economic asset universe

The universe is defined by the underlying economic market rather than by any broker-specific naming convention.

Each market has:

- candidate signal proxies;
- candidate prototype P&L proxies;
- notes where the initial data source is incomplete or economically imperfect.

Unavailable markets remain in the manifest for later data-source expansion.

In [ ]:
# ============================================================
# 3) MASTER ECONOMIC ASSET UNIVERSE
# ============================================================

rows = []


def add_market(
    category,
    market,
    enabled=True,
    signal_candidates="",
    pnl_candidates="",
    notes="",
    universe_start="",
):
    rows.append(
        {
            "category": category,
            "market": market,
            "enabled": bool(enabled),
            "signal_candidates": signal_candidates,
            "pnl_candidates": pnl_candidates,
            "notes": notes,
            "universe_start": universe_start,
        }
    )


# ------------------------------------------------------------
# EQUITY INDICES
# ------------------------------------------------------------

add_market("INDICES", "S&P 500", True, "^GSPC;SPY", "SPY",
           "Price index for signal; adjusted ETF is a prototype dividend-aware P&L proxy.")
add_market("INDICES", "Nasdaq 100", True, "^NDX;QQQ", "QQQ",
           "Price index for signal; adjusted ETF for prototype P&L.")
add_market("INDICES", "DAX", True, "^GDAXI", "EXS1.DE;EWG",
           "Verify final total-return convention.")
add_market("INDICES", "Nikkei 225", True, "^N225", "1321.T;EWJ",
           "ETF is a prototype economic-return proxy.")
add_market("INDICES", "FTSE 100", True, "^FTSE", "ISF.L;EWU", "")
add_market("INDICES", "Russell 2000", True, "^RUT;IWM", "IWM", "")
add_market("INDICES", "CAC 40", True, "^FCHI", "CAC.PA;EWQ", "")
add_market("INDICES", "Hang Seng", True, "^HSI", "2800.HK;EWH", "")
add_market("INDICES", "VIX", True, "^VIX", "",
           "State variable rather than an initial directional portfolio market.")
add_market("INDICES", "S&P/ASX 200", True, "^AXJO", "IOZ.AX;EWA", "")
add_market("INDICES", "IBEX 35", True, "^IBEX", "EWP",
           "Country ETF is not an exact index replication.")
add_market("INDICES", "Swiss Market Index", True, "^SSMI", "EWL", "")
add_market("INDICES", "AEX", True, "^AEX", "IAEX.AS;EWN", "")
add_market("INDICES", "Hang Seng TECH", True, "^HSTECH;3033.HK", "3033.HK",
           "Historical depth must be checked.")
add_market("INDICES", "OMX Stockholm 30", True, "^OMX", "EWD",
           "Proxy mapping must be validated.")
add_market("INDICES", "South Africa 40", True, "^J203.JO;STX40.JO", "EZA",
           "Index availability may be limited.")
add_market("INDICES", "EURO STOXX Volatility", False, "^V2TX", "",
           "Retained in manifest; prototype source may be unavailable.")
add_market("INDICES", "Hang Seng China Enterprises", True, "^HSCE;2828.HK", "2828.HK;FXI", "")
add_market("INDICES", "Straits Times Index", True, "^STI", "ES3.SI;EWS", "")


# ------------------------------------------------------------
# FX
# ------------------------------------------------------------

fx_markets = [
    ("GBP/USD", "GBPUSD=X"),
    ("EUR/USD", "EURUSD=X"),
    ("USD/JPY", "JPY=X"),
    ("AUD/USD", "AUDUSD=X"),
    ("USD/CAD", "CAD=X"),
    ("EUR/GBP", "EURGBP=X"),
    ("USD/CHF", "CHF=X"),
    ("EUR/JPY", "EURJPY=X"),
    ("EUR/CHF", "EURCHF=X"),
    ("US Dollar Index", "DX-Y.NYB"),
]

for market, proxy in fx_markets:
    add_market(
        "FX",
        market,
        True,
        proxy,
        proxy,
        "Spot prototype excludes interest-rate carry.",
    )



# ------------------------------------------------------------
# DIGITAL ASSETS
# ------------------------------------------------------------

add_market(
    "DIGITAL_ASSETS",
    "BTC-USD",
    True,
    "BTC-USD",
    "BTC-USD",
    (
        "Digital assets enter the V2 universe only from 2017-01-01. "
        "Book 02 applies the full conventional-trend warm-up before BTC "
        "can contribute formal transition events. Kept as a separate asset "
        "class so results can be reported with and without digital assets."
    ),
    DIGITAL_ASSETS_START,
)


# ------------------------------------------------------------
# COMMODITIES
# ------------------------------------------------------------

commodity_markets = [
    ("WTI Crude Oil", "CL=F"),
    ("Brent Crude Oil", "BZ=F"),
    ("Natural Gas", "NG=F"),
    ("RBOB Gasoline", "RB=F"),
    ("Gold", "GC=F"),
    ("Silver", "SI=F"),
    ("Platinum", "PL=F"),
    ("Copper", "HG=F"),
    ("Aluminium", "ALI=F"),
    ("Zinc", ""),
    ("Lead", ""),
    ("Iron Ore", ""),
    ("Nickel", ""),
    ("Cocoa", "CC=F"),
    ("Coffee", "KC=F"),
    ("Sugar No. 11", "SB=F"),
    ("Corn", "ZC=F"),
    ("Soybeans", "ZS=F"),
    ("Cotton", "CT=F"),
    ("Soybean Oil", "ZL=F"),
    ("Live Cattle", "LE=F"),
]

for market, proxy in commodity_markets:
    if proxy:
        note = (
            "Prototype continuous futures series; "
            "roll and collateral economics require later treatment."
        )
    else:
        note = (
            "No dependable prototype proxy assigned yet; "
            "retain for later data-source expansion."
        )

    add_market(
        "COMMODITIES",
        market,
        bool(proxy),
        proxy,
        proxy,
        note,
    )


# ------------------------------------------------------------
# BONDS AND RATES
# ------------------------------------------------------------

rates_markets = [
    ("UK 10Y Gilt / Yield", "", "",
     "Needs dedicated source; yields and bond-price returns must not be mixed."),
    ("US Ultra Treasury Bond", "UB=F", "UB=F", ""),
    ("US 30Y Treasury Bond", "ZB=F", "ZB=F", ""),
    ("German Bund", "FGBL=F", "",
     "Prototype availability uncertain; later institutional source preferred."),
    ("Euro-Buxl", "", "", "Dedicated futures source needed."),
    ("US 10Y Treasury Note", "ZN=F", "ZN=F", ""),
    ("Japanese Government Bond", "", "", "Dedicated futures source needed."),
    ("US 5Y Treasury Note", "ZF=F", "ZF=F", ""),
    ("Italian BTP", "", "", "Dedicated futures source needed."),
    ("US 2Y Treasury Note", "ZT=F", "ZT=F", ""),
    ("French OAT", "", "", "Dedicated futures source needed."),
    ("German Bobl", "", "", "Dedicated futures source needed."),
    ("Italian Short-Term BTP", "", "", "Dedicated futures source needed."),
    ("German Schatz", "", "", "Dedicated futures source needed."),
    ("SONIA 3M", "", "", "Dedicated futures source needed."),
    ("Sterling Short-Rate Future", "", "", "Dedicated futures source needed."),
    ("Euribor / ICE Rate Future", "", "", "Dedicated futures source needed."),
    ("30-Day Fed Funds", "ZQ=F", "ZQ=F",
     "Economic-return convention requires validation."),
    ("20+ Year Treasury ETF", "TLT", "TLT",
     "Adjusted ETF useful as a dividend-aware prototype."),
]

for market, signal_proxy, pnl_proxy, note in rates_markets:
    add_market(
        "BONDS_RATES",
        market,
        bool(signal_proxy),
        signal_proxy,
        pnl_proxy,
        note,
    )


universe = pd.DataFrame(rows)

print("Universe created.")
print()

display(
    universe.groupby("category")
    .size()
    .rename("n_markets")
    .to_frame()
)

display(universe)

Universe created.



,n_markets
category,
BONDS_RATES,19
COMMODITIES,21
DIGITAL_ASSETS,1
FX,10
INDICES,19


,category,market,enabled,signal_candidates,pnl_candidates,notes,universe_start
0,INDICES,S&P 500,True,^GSPC;SPY,SPY,Price index for signal; adjusted ETF is a prot...,
1,INDICES,Nasdaq 100,True,^NDX;QQQ,QQQ,Price index for signal; adjusted ETF for proto...,
2,INDICES,DAX,True,^GDAXI,EXS1.DE;EWG,Verify final total-return convention.,
3,INDICES,Nikkei 225,True,^N225,1321.T;EWJ,ETF is a prototype economic-return proxy.,
4,INDICES,FTSE 100,True,^FTSE,ISF.L;EWU,,
5,INDICES,Russell 2000,True,^RUT;IWM,IWM,,
6,INDICES,CAC 40,True,^FCHI,CAC.PA;EWQ,,
7,INDICES,Hang Seng,True,^HSI,2800.HK;EWH,,
8,INDICES,VIX,True,^VIX,,State variable rather than an initial directio...,
9,INDICES,S&P/ASX 200,True,^AXJO,IOZ.AX;EWA,,


In [ ]:
# ============================================================
# 4) SCOPE CHECK AND MANIFEST
# ============================================================

allowed_categories = {
    "INDICES",
    "FX",
    "COMMODITIES",
    "BONDS_RATES",
    "DIGITAL_ASSETS",
}

actual_categories = set(universe["category"])
unexpected_categories = actual_categories.difference(allowed_categories)

if unexpected_categories:
    raise ValueError(
        "Unexpected categories in V2.01 universe: "
        + str(sorted(unexpected_categories))
    )

if "STOCKS" in actual_categories:
    raise ValueError("Individual stocks must not be present in V2.01.")

manifest_path = DIRS["manifests"] / "v2_master_universe.csv"
universe.to_csv(manifest_path, index=False)

print("Scope check passed.")
print("Included: equity indices, FX, commodities, bonds and rates, digital assets.")
print("Total economic markets:", len(universe))
print("Enabled prototype markets:", int(universe["enabled"].sum()))
print("Awaiting a better data source:", int((~universe["enabled"]).sum()))
print("Manifest:", manifest_path)

Scope check passed.
Included: equity indices, FX, commodities, bonds and rates, digital assets.
Total economic markets: 70
Enabled prototype markets: 54
Awaiting a better data source: 16
Manifest: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.01/manifests/v2_master_universe.csv


## Signal prices versus prototype P&L prices

These concepts are deliberately kept separate.

**Signal series** are used later for trend, divergence, SuperbCommand and HMM features.

**P&L series** are used to estimate the economic return from holding a position.

They can differ because dividends, coupons, FX carry, futures roll and collateral returns are not always contained in quoted price histories.

In [ ]:
# ============================================================
# 5) DOWNLOAD HELPERS
# ============================================================

def parse_candidates(value):
    if value is None:
        return []

    if isinstance(value, float) and np.isnan(value):
        return []

    return [
        symbol.strip()
        for symbol in str(value).split(";")
        if symbol.strip()
    ]


def download_one(symbol, start=DATA_START, end=DOWNLOAD_END):
    if not symbol:
        return None

    try:
        data = yf.download(
            symbol,
            start=start,
            end=end,
            auto_adjust=False,
            progress=False,
            actions=True,
            threads=False,
        )

        if data is None or data.empty:
            return None

        if isinstance(data.columns, pd.MultiIndex):
            data.columns = [column[0] for column in data.columns]

        data.index = pd.to_datetime(data.index)

        if data.index.tz is not None:
            data.index = data.index.tz_localize(None)

        data = data[~data.index.duplicated(keep="last")]
        data = data.sort_index()

        return data

    except Exception as exc:
        print(f"Download failed for {symbol}: {exc}")
        return None


def choose_first_available(candidates, min_obs=250):
    diagnostics = []

    for symbol in candidates:
        data = download_one(symbol)

        if data is None:
            observations = 0
        else:
            observations = len(data.dropna(how="all"))

        diagnostics.append((symbol, observations))

        if data is not None and observations >= min_obs:
            return symbol, data, diagnostics

    return None, None, diagnostics


def extract_signal_price(data):
    if data is None:
        return None

    if "Close" not in data.columns:
        return None

    return data["Close"].astype(float)


def extract_pnl_price(data):
    if data is None:
        return None

    if "Adj Close" in data.columns:
        adjusted = data["Adj Close"]

        if adjusted.notna().sum() > 0:
            return adjusted.astype(float)

    if "Close" in data.columns:
        return data["Close"].astype(float)

    return None


print("Download helpers ready.")

Download helpers ready.


In [ ]:
# ============================================================
# 6) PROBE PROTOTYPE DATA COVERAGE
# ============================================================

RUN_COVERAGE_PROBE = True

coverage_rows = []
raw_cache = {}

if RUN_COVERAGE_PROBE:
    probe_universe = universe.loc[universe["enabled"]].copy()

    for _, row in probe_universe.iterrows():
        signal_candidates = parse_candidates(row["signal_candidates"])
        pnl_candidates = parse_candidates(row["pnl_candidates"])

        signal_symbol, signal_data, signal_diagnostics = (
            choose_first_available(signal_candidates)
        )

        pnl_symbol, pnl_data, pnl_diagnostics = (
            choose_first_available(pnl_candidates)
        )

        if signal_symbol is not None and signal_data is not None:
            raw_cache[("signal", row["market"])] = signal_data

        if pnl_symbol is not None and pnl_data is not None:
            raw_cache[("pnl", row["market"])] = pnl_data

        coverage_rows.append(
            {
                "category": row["category"],
                "market": row["market"],
                "universe_start": row.get("universe_start", ""),
                "signal_symbol": signal_symbol,
                "signal_obs": 0 if signal_data is None else len(signal_data),
                "signal_start": (
                    None
                    if signal_data is None
                    else signal_data.index.min().date()
                ),
                "signal_end": (
                    None
                    if signal_data is None
                    else signal_data.index.max().date()
                ),
                "pnl_symbol": pnl_symbol,
                "pnl_obs": 0 if pnl_data is None else len(pnl_data),
                "pnl_start": (
                    None
                    if pnl_data is None
                    else pnl_data.index.min().date()
                ),
                "pnl_end": (
                    None
                    if pnl_data is None
                    else pnl_data.index.max().date()
                ),
                "signal_probe": str(signal_diagnostics),
                "pnl_probe": str(pnl_diagnostics),
            }
        )

coverage = pd.DataFrame(coverage_rows)

if coverage.empty:
    print("Coverage probe produced no rows.")

else:
    coverage["signal_ok"] = coverage["signal_symbol"].notna()
    coverage["pnl_ok"] = coverage["pnl_symbol"].notna()

    coverage = coverage.sort_values(
        ["signal_ok", "category", "market"],
        ascending=[True, True, True],
    )

    display(coverage)

    print(
        "Signal coverage:",
        f"{coverage['signal_ok'].mean():.1%}",
    )

    print(
        "P&L coverage:",
        f"{coverage['pnl_ok'].mean():.1%}",
    )

    coverage_path = (
        DIRS["manifests"] / "v2_01_prototype_coverage.csv"
    )

    coverage.to_csv(coverage_path, index=False)

    print("Coverage report:", coverage_path)

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ^HSTECH"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['^HSTECH']: YFTzMissingError('possibly delisted; no timezone found')
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: FGBL=F"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['FGBL=F']: YFTzMissingError('possibly delisted; no timezone found')


,category,market,universe_start,signal_symbol,signal_obs,signal_start,signal_end,pnl_symbol,pnl_obs,pnl_start,pnl_end,signal_probe,pnl_probe,signal_ok,pnl_ok
48,BONDS_RATES,German Bund,,None,0,None,None,None,0,None,None,"[('FGBL=F', 0)]",[],False,False
53,BONDS_RATES,20+ Year Treasury ETF,,TLT,5895,2002-07-30,2025-12-31,TLT,5895,2002-07-30,2025-12-31,"[('TLT', 5895)]","[('TLT', 5895)]",True,True
52,BONDS_RATES,30-Day Fed Funds,,ZQ=F,6344,2000-09-01,2025-12-31,ZQ=F,6344,2000-09-01,2025-12-31,"[('ZQ=F', 6344)]","[('ZQ=F', 6344)]",True,True
49,BONDS_RATES,US 10Y Treasury Note,,ZN=F,6348,2000-09-21,2025-12-31,ZN=F,6348,2000-09-21,2025-12-31,"[('ZN=F', 6348)]","[('ZN=F', 6348)]",True,True
51,BONDS_RATES,US 2Y Treasury Note,,ZT=F,6414,2000-06-02,2025-12-31,ZT=F,6414,2000-06-02,2025-12-31,"[('ZT=F', 6414)]","[('ZT=F', 6414)]",True,True
47,BONDS_RATES,US 30Y Treasury Bond,,ZB=F,6354,2000-09-21,2025-12-31,ZB=F,6354,2000-09-21,2025-12-31,"[('ZB=F', 6354)]","[('ZB=F', 6354)]",True,True
50,BONDS_RATES,US 5Y Treasury Note,,ZF=F,6360,2000-09-21,2025-12-31,ZF=F,6360,2000-09-21,2025-12-31,"[('ZF=F', 6360)]","[('ZF=F', 6360)]",True,True
46,BONDS_RATES,US Ultra Treasury Bond,,UB=F,4018,2010-01-11,2025-12-31,UB=F,4018,2010-01-11,2025-12-31,"[('UB=F', 4018)]","[('UB=F', 4018)]",True,True
37,COMMODITIES,Aluminium,,ALI=F,2896,2014-05-06,2025-12-31,ALI=F,2896,2014-05-06,2025-12-31,"[('ALI=F', 2896)]","[('ALI=F', 2896)]",True,True
30,COMMODITIES,Brent Crude Oil,,BZ=F,4585,2007-07-30,2025-12-31,BZ=F,4585,2007-07-30,2025-12-31,"[('BZ=F', 4585)]","[('BZ=F', 4585)]",True,True


Signal coverage: 98.1%
P&L coverage: 96.3%
Coverage report: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.01/manifests/v2_01_prototype_coverage.csv


In [ ]:
# ============================================================
# 7) BUILD SIGNAL AND P&L PANELS
# ============================================================

def build_panel(kind):
    if coverage.empty:
        return pd.DataFrame()

    collected = {}

    for _, row in coverage.iterrows():
        symbol = row[f"{kind}_symbol"]

        if pd.isna(symbol):
            continue

        data = raw_cache.get((kind, row["market"]))

        if data is None:
            data = download_one(symbol)

        if data is None:
            continue

        if kind == "signal":
            series = extract_signal_price(data)

        elif kind == "pnl":
            series = extract_pnl_price(data)

        else:
            raise ValueError("kind must be 'signal' or 'pnl'")

        if series is None:
            continue

        series.name = row["market"]
        collected[row["market"]] = series

    if not collected:
        return pd.DataFrame()

    panel = pd.concat(
        list(collected.values()),
        axis=1,
    ).sort_index()

    panel.columns = list(collected.keys())

    return panel


signal_prices = build_panel("signal")
pnl_prices = build_panel("pnl")

print("Signal panel shape:", signal_prices.shape)
print("P&L panel shape:", pnl_prices.shape)

display(signal_prices.tail())

Signal panel shape: (10571, 53)
P&L panel shape: (10571, 52)


,20+ Year Treasury ETF,30-Day Fed Funds,US 10Y Treasury Note,US 2Y Treasury Note,US 30Y Treasury Bond,US 5Y Treasury Note,US Ultra Treasury Bond,Aluminium,Brent Crude Oil,Cocoa,Coffee,Copper,Corn,Cotton,Gold,Live Cattle,Natural Gas,Platinum,RBOB Gasoline,Silver,Soybean Oil,Soybeans,Sugar No. 11,WTI Crude Oil,BTC-USD,AUD/USD,EUR/CHF,EUR/GBP,EUR/JPY,EUR/USD,GBP/USD,US Dollar Index,USD/CAD,USD/CHF,USD/JPY,AEX,CAC 40,DAX,FTSE 100,Hang Seng,Hang Seng China Enterprises,Hang Seng TECH,IBEX 35,Nasdaq 100,Nikkei 225,OMX Stockholm 30,Russell 2000,S&P 500,S&P/ASX 200,South Africa 40,Straits Times Index,Swiss Market Index,VIX
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2025-12-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,87802.156250,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,87835.835938,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-29,88.070000,96.277496,112.718750,104.324219,115.87500,109.460938,118.500,2835.50,61.939999,6242.0,352.149994,5.4905,442.25,64.349998,4325.100098,228.899994,4.687,2104.800049,1.7152,69.856003,48.779999,1049.50,15.26,58.080002,87138.140625,0.67141,0.92900,0.87216,184.240005,1.177274,1.349728,98.040001,1.36683,0.78924,156.462997,946.159973,8112.020020,24351.119141,9866.500000,25635.230469,8891.709961,5.37,17195.800781,25525.560547,50526.921875,2863.639893,2519.800049,6905.740234,8725.700195,116047.898438,4633.640137,13240.589844,14.20
2025-12-30,87.860001,96.277496,112.640625,104.359375,115.81250,109.468750,118.375,2899.25,61.919998,6063.0,350.200012,5.7275,440.50,64.320000,4370.100098,231.175003,3.972,2232.300049,1.7210,77.374001,48.930000,1046.25,14.84,57.950001,88430.132812,0.66955,0.92877,0.87140,183.690002,1.177288,1.351099,98.239998,1.36872,0.78870,156.013000,951.270020,8168.149902,24490.410156,9940.700195,25854.599609,8991.019531,5.46,17354.900391,25462.560547,50339.480469,2882.969971,2500.590088,6896.240234,8717.099609,116501.203125,4655.379883,13267.480469,14.33
2025-12-31,87.160004,96.277496,112.437500,104.386719,115.59375,109.343750,118.000,2906.25,60.849998,6065.0,348.750000,5.6300,440.25,64.269997,4325.600098,232.000000,3.686,2034.500000,1.7054,70.134003,48.070000,1030.50,15.01,57.419998,87508.828125,0.66980,0.93006,0.87224,183.720001,1.174729,1.346720,98.279999,1.36947,0.79170,156.412994,951.289978,8149.500000,NaN,9931.400391,25630.539062,8913.679688,5.39,17307.800781,25249.849609,NaN,NaN,2481.909912,6845.500000,8714.299805,115832.296875,4646.209961,NaN,14.95


In [ ]:
# ============================================================
# 8) DATA-QUALITY REPORT
# ============================================================

def build_quality_report(prices):
    report_rows = []

    for market in prices.columns:
        series = prices[market].dropna()

        if series.empty:
            continue

        returns = series.pct_change()

        report_rows.append(
            {
                "market": market,
                "start": series.index.min().date(),
                "end": series.index.max().date(),
                "observations": len(series),
                "missing_pct_full_panel": prices[market].isna().mean(),
                "zero_return_pct": returns.eq(0).mean(),
                "max_abs_daily_return": returns.abs().max(),
                "median_abs_daily_return": returns.abs().median(),
            }
        )

    if not report_rows:
        return pd.DataFrame()

    return (
        pd.DataFrame(report_rows)
        .sort_values("market")
        .reset_index(drop=True)
    )


signal_quality = build_quality_report(signal_prices)

display(signal_quality)

quality_path = DIRS["manifests"] / "v2_01_signal_quality.csv"
signal_quality.to_csv(quality_path, index=False)

print("Quality report:", quality_path)

,market,start,end,observations,missing_pct_full_panel,zero_return_pct,max_abs_daily_return,median_abs_daily_return
0,20+ Year Treasury ETF,2002-07-30,2025-12-31,5895,0.442342,0.005259,0.075196,0.005356
1,30-Day Fed Funds,2000-09-01,2025-12-31,6344,0.399868,0.610656,0.010253,0.000000
2,AEX,1992-10-12,2025-12-31,8477,0.198089,0.001180,0.107526,0.006050
3,AUD/USD,2006-05-16,2025-12-31,5106,0.516980,0.004700,0.094182,0.004033
4,Aluminium,2014-05-06,2025-12-31,2896,0.726043,0.165746,0.182022,0.005874
5,BTC-USD,2014-09-17,2025-12-31,4124,0.609876,0.000242,0.371695,0.014427
6,Brent Crude Oil,2007-07-30,2025-12-31,4585,0.566266,0.005453,0.244036,0.010961
7,CAC 40,1990-03-01,2025-12-31,9101,0.139060,0.001758,0.122768,0.006797
8,Cocoa,2000-01-03,2025-12-31,6520,0.383218,0.014571,0.229388,0.011261
9,Coffee,2000-01-03,2025-12-31,6518,0.383407,0.011200,0.180943,0.012226


Quality report: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.01/manifests/v2_01_signal_quality.csv


In [ ]:
# ============================================================
# 9) SAVE PROCESSED PANELS
# ============================================================

signal_path = (
    DIRS["data_processed"]
    / "v2_01_signal_prices.parquet"
)

pnl_path = (
    DIRS["data_processed"]
    / "v2_01_pnl_prices_prototype.parquet"
)

signal_prices.to_parquet(signal_path)
pnl_prices.to_parquet(pnl_path)

print("Saved signal panel:")
print(signal_path)

print()
print("Saved prototype P&L panel:")
print(pnl_path)

Saved signal panel:
/content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.01/data/processed/v2_01_signal_prices.parquet

Saved prototype P&L panel:
/content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.01/data/processed/v2_01_pnl_prices_prototype.parquet


# V2.01 completion gate

Expected outputs:

- `v2.01/config/v2_01_config.json`
- `v2.01/manifests/v2_master_universe.csv`
- `v2.01/manifests/v2_01_prototype_coverage.csv`
- `v2.01/manifests/v2_01_signal_quality.csv`
- `v2.01/data/processed/v2_01_signal_prices.parquet`
- `v2.01/data/processed/v2_01_pnl_prices_prototype.parquet`

Before notebook 02, review markets that require stronger data sources, particularly European/UK/Japanese rates and several industrial commodities.

## Research Outcome

Book 01 established the data architecture for the Multi-Asset CTA Strategy V2 research programme. A broad cross-asset universe was successfully assembled across equity indices, FX, commodities, bonds/rates and Bitcoin, while explicitly separating **signal series** from **economic P&L series** where the two are not equivalent.

The resulting framework recognises that price indices, spot FX and continuous futures are suitable research inputs for signal construction but do not necessarily represent complete investable returns because of dividends, interest-rate carry, collateral returns and futures roll effects. This distinction is preserved for later portfolio implementation rather than implicitly treating all downloaded price histories as total-return assets.

The final research universe contains 53 usable markets spanning:

- 18 equity indices / equity-market state series;
- 17 commodities;
- 10 FX markets;
- 7 bonds/rates markets;
- 1 digital asset (BTC-USD).

Individual equities were excluded from the present V2 research universe. Bitcoin was retained as a separate `DIGITAL_ASSETS` asset class, with 1 January 2017 used as its formal research admission date while earlier observations are retained solely for indicator warm-up.

A further methodological principle established in Book 01 is the distinction between **available historical data** and **formally admissible research observations**. Pre-research history may be used to warm long-horizon indicators without allowing those observations to enter subsequent event statistics or model evaluation.

### Conclusion

Book 01 provides a reproducible, cross-asset data foundation for the transition-strategy research programme. Its principal contribution is architectural rather than predictive: it creates a common research panel while preserving the economic distinctions required for later conversion from signal research into an investable CTA portfolio.

**Status: FROZEN as the V2 universe and prototype data-engine foundation.**